# Bipartite Model — 3-Fold CV + Kaggle Submission

Training the Bipartite LR→HR super-resolution model on the brain graph
super-resolution task, with 3-fold cross-validation, full metric evaluation,
and Kaggle submission.

In [1]:
import sys
from pathlib import Path

notebook_dir = Path('.').resolve()
repo_root = notebook_dir.parent
sys.path.insert(0, str(repo_root))

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import KFold

from models.bipartite import GraphSRModel, GraphSRConfig, GraphSRLoss
from utils.matrix_vectorizer import MatrixVectorizer
from utils.metrics import evaluate_fold
from utils.plotting import plot_folds

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
print(f'Device: {DEVICE}')

/Users/dhruvhimatsingka/Desktop/Deep Graph Based Learning/neurores-gnn/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


In [11]:

# Force reload of modules to pick up latest changes
import importlib
import sys

# Remove cached modules
for mod in list(sys.modules.keys()):
    if 'bipartite' in mod or 'matrix_vectorizer' in mod:
        del sys.modules[mod]

# Re-import
from models.bipartite import GraphSRModel, GraphSRConfig, GraphSRLoss
from utils.matrix_vectorizer import MatrixVectorizer

print("✓ Modules reloaded")


✓ Modules reloaded


## Load Data

In [3]:
def load_csv(path):
    arr = np.loadtxt(path, delimiter=',', skiprows=1, dtype=np.float32)
    return arr if arr.ndim > 1 else arr[None, :]

X_lr_train = load_csv(repo_root / 'data' / 'lr_train.csv')
Y_hr_train = load_csv(repo_root / 'data' / 'hr_train.csv')
X_lr_test  = load_csv(repo_root / 'data' / 'lr_test.csv')

print(f'Train LR: {X_lr_train.shape}')
print(f'Train HR: {Y_hr_train.shape}')
print(f'Test  LR: {X_lr_test.shape}')

Train LR: (167, 12720)
Train HR: (167, 35778)
Test  LR: (112, 12720)


## Config

In [ ]:
N_LR, N_HR = 160, 268

# Model configuration
cfg = GraphSRConfig(
    lr_n=N_LR,
    hr_n=N_HR,
    posenc_k=8,
    d_model=128,
    lr_enc_layers=3,
    dropout=0.1,
    bridge_layers=4,
    heads=4,
    dec_hidden=256,
    out_activation='softplus',
)

# Training hyperparameters
EPOCHS = 5
PATIENCE = 30
BATCH_SIZE = 16
LR = 1e-3
WEIGHT_DECAY = 1e-5
STRENGTH_WEIGHT = 0.1  # Optional: weight for strength loss component

print(f'Model config: {cfg}')
print(f'Training: {EPOCHS} epochs (patience={PATIENCE}), bs={BATCH_SIZE}, lr={LR}')

Model config: GraphSRConfig(lr_n=160, hr_n=268, posenc_k=8, d_model=128, lr_enc_layers=3, dropout=0.1, bridge_layers=4, heads=4, dec_hidden=256, out_activation='softplus')
Training: 400 epochs (patience=30), bs=16, lr=0.001


## Training Loop (single fold)

In [5]:
def to_tensor(x, device):
    return torch.from_numpy(x).float().to(device)

def train_one_fold(X_tr, Y_tr, X_va, Y_va, fold_id):
    Xtr, Ytr = to_tensor(X_tr, DEVICE), to_tensor(Y_tr, DEVICE)
    Xva, Yva = to_tensor(X_va, DEVICE), to_tensor(Y_va, DEVICE)

    tr_loader = DataLoader(TensorDataset(Xtr, Ytr), batch_size=BATCH_SIZE, shuffle=True)
    va_loader = DataLoader(TensorDataset(Xva, Yva), batch_size=BATCH_SIZE, shuffle=False)

    model = GraphSRModel(cfg).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = GraphSRLoss(hr_n=N_HR, strength_weight=STRENGTH_WEIGHT)

    best_val, best_state = float('inf'), None
    patience_counter = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_losses = []
        for x_vec, y_vec in tr_loader:
            y_hat = model(x_vec)
            loss = loss_fn(y_hat, y_vec)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            train_losses.append(loss.item())

        train_loss = np.mean(train_losses) if len(train_losses) > 0 else float('nan')

        model.eval()
        val_losses = []
        with torch.no_grad():
            for x_vec, y_vec in va_loader:
                y_hat = model(x_vec)
                val_losses.append(loss_fn(y_hat, y_vec).item())

        val_loss = np.mean(val_losses) if len(val_losses) > 0 else float('nan')
        improved = val_loss < best_val
        if improved:
            best_val = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if epoch % 10 == 0 or epoch == 1:
            mark = ' *' if improved else ''
            print(f'  Fold {fold_id} | Epoch {epoch:3d}/{EPOCHS} | Train: {train_loss:.6f} | Val: {val_loss:.6f}{mark}')

        if patience_counter >= PATIENCE:
            print(f'  Early stopping at epoch {epoch} (no improvement for {PATIENCE} epochs)')
            break

    model.load_state_dict(best_state)
    print(f'  Fold {fold_id} best val loss: {best_val:.6f}')
    return model

## 3-Fold Cross-Validation + Metrics

In [12]:
kf = KFold(n_splits=3, shuffle=True, random_state=SEED)
vectorizer = MatrixVectorizer()

fold_metrics = []
fold_models = []

for fold_id, (tr_idx, va_idx) in enumerate(kf.split(X_lr_train), start=1):
    print(f"\n{'='*50}")
    print(f'Fold {fold_id}: train={len(tr_idx)}, val={len(va_idx)}')
    print(f"{'='*50}")

    model = train_one_fold(
        X_lr_train[tr_idx], Y_hr_train[tr_idx],
        X_lr_train[va_idx], Y_hr_train[va_idx],
        fold_id,
    )
    fold_models.append(model)

    # Evaluate on validation set
    model.eval()
    va_preds, va_targets = [], []
    va_loader = DataLoader(
        TensorDataset(to_tensor(X_lr_train[va_idx], DEVICE),
                       to_tensor(Y_hr_train[va_idx], DEVICE)),
        batch_size=BATCH_SIZE, shuffle=False,
    )
    with torch.no_grad():
        for x_vec, y_vec in va_loader:
            y_hat = model(x_vec).detach().cpu().numpy()
            va_preds.append(y_hat)
            va_targets.append(y_vec.detach().cpu().numpy())

    pred_vecs = np.concatenate(va_preds)
    gt_vecs = np.concatenate(va_targets)
    pred_mats = np.stack([vectorizer.anti_vectorize(v, N_HR) for v in pred_vecs])
    gt_mats = np.stack([vectorizer.anti_vectorize(v, N_HR) for v in gt_vecs])

    print(f'\n  Metrics for fold {fold_id}:')
    metrics = evaluate_fold(pred_mats, gt_mats, verbose=True)
    fold_metrics.append(metrics)

print('\n\nAll 3 folds evaluated.')


Fold 1: train=111, val=56
  Fold 1 | Epoch   1/400 | Train: 243.970295 | Val: 42.331491 *


KeyboardInterrupt: 

## Bar Plots

In [ ]:
plot_folds(fold_metrics)

## Generate Kaggle Submission (ensemble of 3 folds)

In [ ]:
test_preds = []
for i, model in enumerate(fold_models):
    model.eval()
    loader = DataLoader(
        TensorDataset(to_tensor(X_lr_test, DEVICE)),
        batch_size=BATCH_SIZE, shuffle=False,
    )
    preds = []
    with torch.no_grad():
        for (x_vec,) in loader:
            y_hat = model(x_vec).detach().cpu().numpy()
            preds.append(y_hat)
    test_preds.append(np.concatenate(preds))
    print(f'Fold {i+1} test predictions: {test_preds[-1].shape}')

ensemble = np.clip(np.mean(test_preds, axis=0), a_min=0.0, a_max=None)
print(f'\nEnsemble shape: {ensemble.shape}')
print(f'Range: [{ensemble.min():.6f}, {ensemble.max():.6f}]')

In [ ]:
import pandas as pd

n_subjects, n_features = ensemble.shape
ids = np.arange(1, n_subjects * n_features + 1)
submission = pd.DataFrame({'ID': ids, 'Predicted': ensemble.flatten()})

sub_dir = repo_root / 'submission'
sub_dir.mkdir(exist_ok=True)

counter = 1
out_path = sub_dir / f'bipartite_submission_{counter}.csv'
while out_path.exists():
    counter += 1
    out_path = sub_dir / f'bipartite_submission_{counter}.csv'

submission.to_csv(out_path, index=False)

print(f'Saved to {out_path} ({len(submission):,} rows)')
print(submission.head())